# P2 · JAAH for both frame and event teachers

Use this notebook instead of 02f. It runs four new jobs: F-control, F-mixed, E-control, E-mixed, then evaluates all four F/E pairings. Read docs/p2/P2_JAAH_EXPERIMENT.md. Enable Internet and GPU. Each job runs 30 epochs; F and E have different native update budgets, matched within each control/mixed pair. No held-out test labels are evaluated.


In [ ]:
from pathlib import Path
import subprocess, sys, os, json
project = Path("/kaggle/working/chord-practice-system")
if not project.exists():
    subprocess.run(["git", "clone", "https://github.com/PhatLavar/chord-practice-system.git", str(project)], check=True)
os.chdir(project)
subprocess.run(["git", "pull", "--ff-only", "origin", "main"], check=True)
print(subprocess.check_output(["git", "rev-parse", "HEAD"], text=True))
import torch, numpy
print("torch", torch.__version__, "numpy", numpy.__version__, "CUDA", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
# Official inputs. Downloaded archives are read without extraction.
from urllib.request import urlretrieve
raw = project / "data/raw"
raw.mkdir(parents=True, exist_ok=True)
sources = {
    "index.csv": "https://www.dropbox.com/s/o0olz0uwl9z9stb/billboard-2.0-index.csv?dl=1",
    "full.tar.xz": "https://www.dropbox.com/s/t390alzrkx0c9yt/billboard-2.0.1-lab.tar.xz?dl=1",
    "billboard-2.0-chordino.tar.xz": "https://www.dropbox.com/s/e9dm23vbawg9dsw/billboard-2.0-chordino.tar.xz?dl=1",
}
for name, url in sources.items():
    if not (raw / name).exists():
        temporary = raw / (name + ".part")
        urlretrieve(url, temporary)
        temporary.replace(raw / name)
cache = project / "data/processed/p1"
if not (cache / "metadata.json").exists():
    subprocess.run([sys.executable, "src/acr/p1_data.py"], check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests"], check=True)


In [ ]:
# Download the pinned official JAAH revision; keep generated audit files in runs/.
aux_cache = project / "data/processed/p2-jaah"
if not aux_cache.exists():
    subprocess.run([sys.executable, "src/acr/prepare_jaah.py", "--download",
                    "--report", "runs/p2-jaah-preparation"], check=True)
# Validate all cache hashes and source identity against the committed audit.
sys.path.insert(0, str(project / "src/acr"))
from experiment_jaah import load_aux
aux_songs, aux_events, aux_meta = load_aux(aux_cache)
print({"JAAH_recordings": len(aux_songs), "events": len(aux_events),
       "revision": aux_meta["revision"], "excluded": len(aux_meta["excluded"])})
del aux_songs, aux_events


In [ ]:
# Reuse only completed stages with the exact config, source code and run mode.
import hashlib
revision = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
source_hashes = {p.as_posix(): hashlib.sha256(p.read_bytes()).hexdigest()
                 for p in Path("src/acr").glob("*.py")}
def run_stage(script, stage, config, smoke=False):
    if stage.exists():
        assert (stage / "comparison.json").exists(), f"Incomplete stage: {stage}; preserve it and choose a fresh run root."
        expected = json.loads(Path(config).read_text())
        for arm in ("billboard_control", "jaah_mixed"):
            record = json.loads((stage / arm / "run.json").read_text())
            assert record["config"] == expected and record["smoke"] == smoke
            assert record["git_revision"] == revision
            assert {k.replace(chr(92), "/"): v for k, v in record["source_code_sha256"].items()} == source_hashes
        print("Reusing completed stage:", stage)
        return
    command = [sys.executable, script, "--config", config, "--output", str(stage),
               "--device", "cpu" if smoke else "cuda"]
    if smoke: command.append("--smoke")
    subprocess.run(command, check=True)
smoke_root = project / "runs/p2-jaah-fe-kaggle-smoke"
run_stage("src/acr/experiment_jaah_f.py", smoke_root / "F", "configs/p1-jaah-pilot.json", smoke=True)
run_stage("src/acr/experiment_jaah.py", smoke_root / "E", "configs/p2-jaah-pilot.json", smoke=True)
if not (smoke_root / "analysis").exists():
    subprocess.run([sys.executable, "src/acr/compare_jaah_pairs.py", "--runs", str(smoke_root),
                    "--output", str(smoke_root / "analysis"), "--smoke"], check=True)


In [ ]:
assert torch.cuda.is_available(), "Enable GPU for the full experiments."
run_root = project / "runs/p2-jaah-fe-pilot"
# Jobs 1 and 2: independently initialized frame teachers.
run_stage("src/acr/experiment_jaah_f.py", run_root / "F", "configs/p1-jaah-pilot.json")


In [ ]:
# Jobs 3 and 4: independently initialized event teachers.
run_stage("src/acr/experiment_jaah.py", run_root / "E", "configs/p2-jaah-pilot.json")


In [ ]:
# Align all four checkpoints on the same gold intervals and compare all four pairs.
analysis = run_root / "analysis"
if not analysis.exists():
    subprocess.run([sys.executable, "src/acr/compare_jaah_pairs.py", "--runs", str(run_root),
                    "--output", str(analysis)], check=True)
assert (analysis / "summary.json").exists(), "Incomplete analysis; inspect before retrying."
print((analysis / "individuals.csv").read_text())
print((analysis / "summary.json").read_text())


In [ ]:
import shutil
archive = shutil.make_archive("/kaggle/working/p2-jaah-fe-results", "zip",
                              project, "runs/p2-jaah-fe-pilot")
print(archive)


Save the executed notebook version and download p2-jaah-fe-results.zip to local runs/imports/. The archive includes four checkpoints, F frame logits and pooled event probabilities, E event logits, per-source training exposure, and all four pair diagnostics (component confidence/precision/coverage, per-class support and complementarity). Smoke scores are not research results. A completed teacher stage may be reused at the same config/revision/source hashes; interrupted stages require inspection and a fresh output root. This does not resume halfway through a job.
